# 🐍 Master Guide: Generators, `yield` & Coroutines
> **Single-Source Revision Guide**: Covers generator suspension, stack frame pausing (`PyGenObject`), two-way coroutine communication (`.send()`, `.throw()`), and `yield from`.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Generators are functions containing `yield`. Instead of returning once and destroying their stack frame, generators pause execution, preserve their local state, and yield values on demand.

---
## 🧠 2. CPython Architecture: How `yield` Suspends Execution

### `PyGenObject` & Frame Preservation
When a function containing `yield` is invoked, CPython does **not** execute its body immediately. It constructs a `PyGenObject` holding a detached execution frame (`gi_frame`):

```
1. gen = my_generator() -> Returns PyGenObject in 'GEN_CREATED' state
2. next(gen)            -> Resumes frame, executes until 'yield value'
                           Frame pauses, state saved, state is 'GEN_SUSPENDED'
3. next(gen)            -> Resumes from line AFTER yield with all locals intact!
4. Return / exit        -> Frame detached, raises StopIteration, state 'GEN_CLOSED'
```

- **Memory Footprint**: Generates infinite series with **$O(1)$ constant memory**!

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Basic Generator Function

In [ ]:
def count_up_to(max_val):
    count = 1
    while count <= max_val:
        yield count
        count += 1

counter = count_up_to(3)
print("Generator object:", counter)
print("First yield :", next(counter))
print("Second yield:", next(counter))
print("Third yield :", next(counter))

# Looping over generator:
for num in count_up_to(5):
    print("Loop item:", num)

---
## 🟡 4. Part 2: Intermediate Features

### Two-Way Communication with `.send(val)`
`yield` is an expression! Values sent via `.send()` become the result of the `yield` expression inside the generator.

In [ ]:
def running_averager():
    total = 0.0
    count = 0
    average = None
    while True:
        # Execution pauses here; .send() injects val into 'term'
        term = yield average
        total += term
        count += 1
        average = total / count

avg = running_averager()
next(avg)  # Prime the generator to the first yield

print("Average after 10:", avg.send(10))
print("Average after 20:", avg.send(20))
print("Average after 30:", avg.send(30))

### Sub-Generator Delegation with `yield from`
Transparently delegates to a nested sub-generator.

In [ ]:
def chain_generators(*iterables):
    for it in iterables:
        yield from it  # Delegates directly!

combined = list(chain_generators([1, 2], ["a", "b"], (True, False)))
print("Chained sequence:", combined)

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: The Unprimed `.send()` TypeError Trap
You cannot send a non-None value to a newly created generator! You MUST prime it first with `next(gen)` or `gen.send(None)`.

In [ ]:
def echo():
    val = yield
    yield val

g = echo()
try:
    g.send(42) # Unprimed!
except TypeError as e:
    print("TypeError caught:", e)
    print("Rule: Always call next(g) or g.send(None) before sending data!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| yield value                        | Suspends execution, returns value     |
| gen.send(val)                      | Injects value into yield expression   |
| yield from subgen                  | Transparently delegates to subgen     |
| Memory                             | O(1) constant memory streaming        |
| Priming                            | Call next(g) before g.send(data)      |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Custom Range Generator with `yield` (`GenEx1.py` - `GenEx3.py`)
> **Core Generator Rules**:
> - Any function containing the `yield` keyword becomes a **Generator Function**.
> - When called, it does not execute immediately; it returns a `<class 'generator'>` object.
> - `yield` pauses function state and returns a value. Calling `next()` resumes from the exact paused line.

In [ ]:
# Practical Implementation: custom_range Generator (GenEx1 - GenEx3)
def custom_range(begin, end=None, step=1):
    """Simulates Python's built-in range() using a generator."""
    if end is None:
        current = 0
        stop = begin
    else:
        current = begin
        stop = end

    while current < stop:
        yield current
        current += step

# Using custom_range
print("custom_range(5):", list(custom_range(5)))
print("custom_range(10, 25, 3):", list(custom_range(10, 25, 3)))

# Inspecting generator memory footprint
gen_obj = custom_range(1_000_000)
import sys
print(f"Memory of 1,000,000 item generator: {sys.getsizeof(gen_obj)} bytes (O(1) Memory!)")

### 2. Generator Pipeline for Data Streaming
Chaining generators to filter, transform, and stream records efficiently without memory overhead.

In [ ]:
# Generator Data Pipeline Drill
def transaction_stream():
    transactions = [
        {"id": 1, "amt": 500, "status": "APPROVED"},
        {"id": 2, "amt": 25000, "status": "PENDING"},
        {"id": 3, "amt": 1200, "status": "APPROVED"},
        {"id": 4, "amt": 8000, "status": "REJECTED"},
        {"id": 5, "amt": 4500, "status": "APPROVED"}
    ]
    for tx in transactions:
        yield tx

def filter_approved(stream):
    for tx in stream:
        if tx["status"] == "APPROVED":
            yield tx

def apply_tax(stream, rate=0.18):
    for tx in stream:
        taxed = dict(tx)
        taxed["tax"] = tx["amt"] * rate
        taxed["total"] = tx["amt"] + taxed["tax"]
        yield taxed

# Running the streamed pipeline
pipeline = apply_tax(filter_approved(transaction_stream()))
print("Processed Transactions Pipeline:")
for item in pipeline:
    print(f"  TX #{item['id']} | Net: Rs. {item['amt']} | Tax: Rs. {item['tax']:.2f} | Total: Rs. {item['total']:.2f}")